# Sanctions Screening

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/sanctions-screening.ipynb)

**What:** Screen names of people, companies and vessels against the official OFAC SDN, OFAC Consolidated, UN, EU and UK sanctions lists. Fuzzy match handles diacritics, transliteration, word order and aliases; optional DOB and country. Screening aid, not legal advice. $0.004 per name.

**Why:** run the [`sanctions-screening`](https://apify.com/transparent_meteorite/sanctions-screening) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Screened names are replaced by 'name #N' and matched entity names are not displayed, so no person names are stored in this notebook's output.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/sanctions-screening) for every option.

In [2]:
run_input = {   'names': ['Rafael Caro-Quintero', 'Mohamed Ali', 'Acme Plumbing Supplies LLC'],
    'lists': ['ofac_sdn', 'un'],
    'minScore': 88,
    'maxMatchesPerName': 3}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/sanctions-screening').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> Status: RUNNING, Message: 


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:49.807Z ACTOR: Pulling container image of build SkDzTdgYNEdB6I7zb from registry.


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:50.615Z ACTOR: Creating container.


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:50.687Z ACTOR: Starting container.


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:50.693Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:54.715Z Done in 3s: 3 screened (2 possible_match, 1 clear), 3 charged, lists failed: none


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:54.782Z INFO  [Status message]: Done: 3 items


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> 2026-10-13T22:25:54.829Z Thanks for using Sanctions Screening. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/pBHSTZBXHWO3zPKby/issues) with the run URL and we will reply there.


[apify.sanctions-screening runId:yWSi7MIpk24SS2cd5] -> Status: SUCCEEDED, Message: Done: 3 items


Run status: SUCCEEDED | items: 3


## 4. Tabulate

Screened names are replaced by 'name #N' and matched entity names are not displayed, so no person names are stored in this notebook's output.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)
df['query_no'] = ['name #%d' % (i + 1) for i in range(len(df))]

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["query_no", "status", "bestScore", "matchCount", "topMatchList", "topMatchPrograms", "listsChecked"] if c in df.columns]
df[SHOW]

,query_no,status,bestScore,matchCount,topMatchList,topMatchPrograms,listsChecked
0,name #1,possible_match,100.0,1,OFAC SDN,[SDNTK],"[un, ofac_sdn]"
1,name #2,possible_match,98.8,40,UN Security Council,[CAR],"[un, ofac_sdn]"
2,name #3,clear,67.1,0,NaN,None,"[un, ofac_sdn]"


## 5. A quick analysis

In [5]:
df['bestScore']=pd.to_numeric(df['bestScore'],errors='coerce')
print(df['status'].value_counts())
print(df[['query_no','status','bestScore']].to_string(index=False))
print('Screening aid only; a possible match is a prompt for human review, not a determination.')

status
possible_match    2
clear             1
Name: count, dtype: int64
query_no         status  bestScore
 name #1 possible_match      100.0
 name #2 possible_match       98.8
 name #3          clear       67.1
Screening aid only; a possible match is a prompt for human review, not a determination.


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [sanctions-screening on the Apify Store](https://apify.com/transparent_meteorite/sanctions-screening).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.